# Exercises week 42

**October 12–16, 2026**

Date: **Deadline is Friday October 16 at midnight**


# Overarching aims of the exercises this week

The aim of the exercises this week is to compute the gradient of the neural
network you implemented last week, and then to train it.

To train neural networks we use gradient descent, since there is no
analytical expression for the optimal parameters. This means that you need
the gradient of the cost function with respect to the network parameters,
and then a gradient method. The gradient is computed with the
**backpropagation algorithm**, Section [8.8](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter8.html#backpropagation) of the
lecture notes, which Monday's lecture derived and which you now write by
hand. At every step you check your result against `jax.grad`, which does the
same computation for you.

You will begin by computing the gradient of a network with one layer, then
two layers, then any number of layers. Keeping track of the shapes and doing
things step by step will be very important this week.

Exercises 2–5 use the **single-vector convention** of exercises 1–4 of last
week: one input vector at a time, weights of shape (outputs, inputs) and
$z = Wx + b$. Exercise 6 switches to the **batched convention** of the lecture
notes (samples in rows, weights of shape (inputs, outputs), $Z = XW + b$,
Eq. (8.21)), which is what you need for Project 2. If you did not finish
last week's set, the full solution is in `exercisesweek41_solutions.ipynb`.

We recommend that you do the exercises by editing and running this notebook,
as it includes some checks along the way that you have implemented the
pieces correctly, and running small parts of the code at a time is important
for understanding the methods. If you cannot run the notebook locally, Google
Colab works as well, though we recommend that you set up VSCode (or
JupyterLab) and your Python environment to run code like this on your own
machine; ask on the course Discord or at the lab sessions if you have
trouble.

First, some setup code that you will need; don't change this cell. As last
week, the activation and cost functions are written with `jax.numpy`
(`jnp`) so that JAX can differentiate them. The derivatives `ReLU_der` and
`sigmoid_der` are the ones you will use in your own backpropagation.


In [1]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)   # 64-bit floats, so that results agree with numpy to ~1e-15
import jax.numpy as jnp                    # numpy-compatible functions that JAX can differentiate
from sklearn import datasets
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

np.random.seed(2026)   # so that your numbers can be compared with those of other students


# Defining some activation functions
def ReLU(z):
    return jnp.where(z > 0, z, 0.0)


# Derivative of the ReLU function
def ReLU_der(z):
    return jnp.where(z > 0, 1.0, 0.0)


def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))


def mse(predict, target):
    return jnp.mean((predict - target) ** 2)

# Exercise 1 - Understand the feed forward pass

**a)** Complete last week's exercises if you haven't already (recommended),
and compare with `exercisesweek41_solutions.ipynb`. In particular make sure
you understand the shapes of `W`, `b`, `z` and `a` in the two conventions
(single vector and batched), since every gradient below has the shape of the
parameter it belongs to.


# Exercise 2 - Gradient with one layer using JAX

For the first few exercises we will not use batched inputs. Only a single
input vector is passed through the layer at a time.

In this exercise you will compute the gradient of a single layer. You only
need to change the code in the cells right below an exercise; the rest works
out of the box. Feel free to make changes and see how things work, though!


**a)** If the weights and bias of a layer have shapes (10, 4) and (10,),
what will the shapes of the gradients of the cost function with respect to
these weights and this bias be?


The shape of the gradients will be (10, 4) and (10,), respectively.

**b)** Complete the `feed_forward_one_layer` function. It should use the
sigmoid activation function. Also define the weight and bias with the
correct shapes.


In [2]:
def feed_forward_one_layer(W, b, x):
    z = W @ x + b
    a = sigmoid(z)
    return a


def cost_one_layer(W, b, x, target):
    predict = feed_forward_one_layer(W, b, x)
    return mse(predict, target)


x = np.random.rand(2)
target = np.random.rand(3)

W = np.random.rand(3,2)
b = np.random.rand(3)

**c)** Compute the gradient of the cost function with respect to the weight
and bias by running the cell below. You will not need to change anything,
just make sure it runs by defining things correctly in the cell above.
`jax.grad(cost_one_layer, argnums=(0, 1))` returns a function that computes
the gradient of `cost_one_layer` with respect to its first two arguments,
`W` and `b`, by reverse-mode automatic differentiation — which for a neural
network *is* backpropagation (Section 4.14 of the lecture notes).


In [3]:
jax_one_layer = jax.grad(cost_one_layer, argnums=(0, 1))
W_g, b_g = jax_one_layer(W, b, x, target)
print(W_g, b_g)

[[-0.01004511 -0.01891421]
 [ 0.01879958  0.03539823]
 [ 0.00751831  0.01415642]] [-0.04579581  0.08570757  0.03427607]


# Exercise 3 - Gradient with one layer writing backpropagation by hand

Before you use the gradient you found using JAX, you will have to find the
gradient "manually", to better understand how the backpropagation
computation works. To do backpropagation "manually", you will need to write
out expressions for many derivatives along the computation. This is the
scalar/one-layer case of Section
[8.8.1](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter8.html#two-scalar-examples-no-hidden-layer-and-one-hidden-layer)
of the lecture notes, now with vectors.


We want to find the gradient of the cost function with respect to the weight
and bias. This is quite hard to do directly, so we instead use the chain rule
to combine multiple derivatives which are easier to compute.

$$
\frac{dC}{dW} = \frac{dC}{da}\frac{da}{dz}\frac{dz}{dW}
$$

$$
\frac{dC}{db} = \frac{dC}{da}\frac{da}{dz}\frac{dz}{db}
$$


**a)** Which intermediary results can be reused between the two expressions?


We may reuse the intermediary result

$$
\frac{dC}{dz} = \frac{dC}{da}\frac{da}{dz},
$$

as it appears in both expressions.

**b)** What is the derivative of the cost with respect to the final
activation? You can use the JAX calculation to make sure you get the correct
result. Remember that we compute the *mean* in `mse`.


The derivative of the cost with respect to the final activation $a$ is

$$
\frac{dC}{da} = \frac{2}{n}(a - y),
$$

where $n$ is the number of features in $a$ (alternatively $y$) and $y$ is the target.

In [4]:
z = W @ x + b
a = sigmoid(z)

predict = a


def mse_der(predict, target):
    return 2*(predict - target) / len(predict)


print(mse_der(predict, target))

cost_jax = jax.grad(mse, argnums=0)
print(cost_jax(predict, target))

[-0.20626037  0.39009356  0.19173103]
[-0.20626037  0.39009356  0.19173103]


**c)** What is the expression for the derivative of the sigmoid activation
function? You can use the JAX calculation to make sure you get the correct
result. (Since the sigmoid acts element by element, the gradient of
`sum(sigmoid(z))` with respect to `z` is exactly the element-wise derivative
$\sigma'(z)$; that is what the JAX line below computes.)


The derivative of the sigmoid activation function is

$$
\frac{da}{dz} = \frac{e^{-z}}{\left(1 + e^{-z}\right)^2} = e^{-z} a^2.
$$

In [5]:
def sigmoid_der(z):
    return jnp.exp(-z) * sigmoid(z)**2


print(sigmoid_der(z))

sigmoid_jax = jax.grad(lambda z: jnp.sum(sigmoid(z)))   # element-wise derivative of an element-wise function
print(sigmoid_jax(z))

[0.22202913 0.2197103  0.17877165]
[0.22202913 0.2197103  0.17877165]


**d)** Using the two derivatives you just computed, compute this
intermediary gradient you will use later (it is the *error* $\delta$ of the
layer, Eq. (8.60) of the lecture notes):

$$
\frac{dC}{dz} = \frac{dC}{da}\frac{da}{dz}
$$


In [6]:
dC_da = mse_der(predict, target)
print(dC_da)

dC_dz = np.outer(dC_da, sigmoid_der(z))

[-0.20626037  0.39009356  0.19173103]


**e)** What is the derivative of the intermediary `z` with respect to the
weight and bias? What should the shapes be? The one for the weights is a
little tricky; it can be easier to play around in the next exercise first.
You can also try computing it with JAX to get a hint. (Lecture notes:
Eqs. (8.61)–(8.63), "the error is the bias gradient", and the outer product
with the layer input for the weights.)


Recall that $z = W x + b$. Hence,

$$
\frac{dz}{dW} =  x, \qquad \frac{dz}{db} = I_{3 \times 3},
$$

where $I_{3 \times 3}$ is the $3 \times 3$ identity matrix.

**f)** Now combine the expressions you have worked with so far to compute
the gradients! Note that you always need to do a feed forward pass while
saving the `z`s and `a`s before you do backpropagation, as they are used in
the derivative expressions.


In [7]:
dC_da = mse_der(predict, target)
dC_dz = dC_da * sigmoid_der(z)
dC_dW = np.outer(dC_dz, x)
dC_db = dC_dz

print(dC_dW, dC_db)

[[-0.01004511 -0.01891421]
 [ 0.01879958  0.03539823]
 [ 0.00751831  0.01415642]] [-0.04579581  0.08570757  0.03427607]


You should get the same results as with JAX.


In [8]:
W_g, b_g = jax_one_layer(W, b, x, target)
print(W_g, b_g)
print(np.allclose(dC_dW, W_g), np.allclose(dC_db, b_g))

[[-0.01004511 -0.01891421]
 [ 0.01879958  0.03539823]
 [ 0.00751831  0.01415642]] [-0.04579581  0.08570757  0.03427607]
True True


# Exercise 4 - Gradient with two layers writing backpropagation by hand


Now that you have implemented backpropagation for one layer, you have found
most of the expressions you will need for more layers. Let's move up to two
layers (compare with the two-layer network of Section
[8.8.2](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter8.html#two-inputs-two-hidden-nodes-one-output) of the lecture
notes).


In [9]:
x = np.random.rand(2)
target = np.random.rand(4)

W1 = np.random.rand(3, 2)
b1 = np.random.rand(3)

W2 = np.random.rand(4, 3)
b2 = np.random.rand(4)

layers = [(W1, b1), (W2, b2)]

In [10]:
z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

We begin by computing the gradients of the last layer, as the gradients
must be propagated backwards from the end.

**a)** Compute the gradients of the last layer, just like you did for the
single layer in the previous exercise.


In [11]:
dC_da2 = ...
dC_dz2 = ...
dC_dW2 = ...
dC_db2 = ...

To find the derivative of the cost with respect to the activation of the
first layer, we need a new expression, the one furthest to the right in the
following.

$$
\frac{dC}{da_1} = \frac{dC}{dz_2}\frac{dz_2}{da_1}
$$

**b)** What is the derivative of the second-layer intermediary with respect
to the first-layer activation? (First recall how you compute $z_2$.) This is
the step that *propagates the error backwards*, Eq. (8.64) of the lecture
notes.

$$
\frac{dz_2}{da_1}
$$


**c)** Use this expression, together with expressions which are equivalent
to the ones for the last layer, to compute all the derivatives of the first
layer.

$$
\frac{dC}{dW_1} = \frac{dC}{da_1}\frac{da_1}{dz_1}\frac{dz_1}{dW_1}
$$

$$
\frac{dC}{db_1} = \frac{dC}{da_1}\frac{da_1}{dz_1}\frac{dz_1}{db_1}
$$


In [12]:
dC_da1 = ...
dC_dz1 = ...
dC_dW1 = ...
dC_db1 = ...

In [13]:
print(dC_dW1, dC_db1)
print(dC_dW2, dC_db2)

Ellipsis Ellipsis
Ellipsis Ellipsis


**d)** Make sure you got the same gradient as the following code, which
uses JAX to do backpropagation. (`jax.grad` accepts a list of tuples of
arrays as the argument to differentiate and returns a gradient with the
same structure.)


In [14]:
def feed_forward_two_layers(layers, x):
    W1, b1 = layers[0]
    z1 = W1 @ x + b1
    a1 = sigmoid(z1)

    W2, b2 = layers[1]
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)

    return a2

In [15]:
def cost_two_layers(layers, x, target):
    predict = feed_forward_two_layers(layers, x)
    return mse(predict, target)


grad_two_layers = jax.grad(cost_two_layers, argnums=0)
grad_two_layers(layers, x, target)

[(Array([[0.00037557, 0.00346971],
         [0.00035952, 0.00332141],
         [0.00013004, 0.00120135]], dtype=float64),
  Array([0.00779593, 0.00746271, 0.00269925], dtype=float64)),
 (Array([[ 0.01022962,  0.01086494,  0.01324558],
         [ 0.01903882,  0.02022125,  0.02465197],
         [ 0.00372743,  0.00395892,  0.00482637],
         [-0.00685045, -0.00727591, -0.00887015]], dtype=float64),
  Array([ 0.01856586,  0.03455379,  0.00676495, -0.01243297], dtype=float64))]

**e)** How would you use the gradient from this layer to compute the
gradient of an even earlier layer? Would the expressions be any different?


# Exercise 5 - Gradient with any number of layers writing backpropagation by hand


Well done on getting this far! Now it's time to compute the gradient with
any number of layers: the four equations of backpropagation, Section
[8.8.4](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter8.html#the-four-equations-of-backpropagation), and the
algorithm of Section [8.8.5](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter8.html#the-algorithm).

First, some code from the general neural network code from last week. Note
that we are still sending in one input vector at a time. We will change it to
use batched inputs later.


In [16]:
def create_layers(network_input_size, layer_output_sizes):
    layers = []

    i_size = network_input_size
    for layer_output_size in layer_output_sizes:
        W = np.random.randn(layer_output_size, i_size)
        b = np.random.randn(layer_output_size)
        layers.append((W, b))

        i_size = layer_output_size
    return layers


def feed_forward(input, layers, activation_funcs):
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        z = W @ a + b
        a = activation_func(z)
    return a


def cost(layers, input, activation_funcs, target):
    predict = feed_forward(input, layers, activation_funcs)
    return mse(predict, target)

You might already have noticed a very important detail in backpropagation:
you need the values from the forward pass to compute all the gradients! The
feed forward function above is great for efficiency and for using JAX, as it
only cares about computing the final output, but now we need to also save
the results along the way.

Here is a function which does that for you.


In [17]:
def feed_forward_saver(input, layers, activation_funcs):
    layer_inputs = []
    zs = []
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        layer_inputs.append(a)
        z = W @ a + b
        a = activation_func(z)

        zs.append(z)

    return layer_inputs, zs, a

**a)** Now complete the backpropagation function so that it returns the
gradient of the cost function with respect to all the weights and biases.
Use the JAX calculation below to make sure you get the correct answer.


In [18]:
def backpropagation(
    input, layers, activation_funcs, target, activation_ders, cost_der=mse_der
):
    layer_inputs, zs, predict = feed_forward_saver(input, layers, activation_funcs)

    layer_grads = [() for layer in layers]

    # We loop over the layers, from the last to the first
    for i in reversed(range(len(layers))):
        layer_input, z, activation_der = layer_inputs[i], zs[i], activation_ders[i]

        if i == len(layers) - 1:
            # For last layer we use cost derivative as dC_da(L) can be computed directly
            dC_da = ...
        else:
            # For other layers we build on previous z derivative, as dC_da(i) = dC_dz(i+1) * dz(i+1)_da(i)
            (W, b) = layers[i + 1]
            dC_da = ...

        dC_dz = ...
        dC_dW = ...
        dC_db = ...

        layer_grads[i] = (dC_dW, dC_db)

    return layer_grads

In [19]:
network_input_size = 2
layer_output_sizes = [3, 4]
activation_funcs = [sigmoid, ReLU]
activation_ders = [sigmoid_der, ReLU_der]

layers = create_layers(network_input_size, layer_output_sizes)

x = np.random.rand(network_input_size)
target = np.random.rand(4)

In [20]:
layer_grads = backpropagation(x, layers, activation_funcs, target, activation_ders)
print(layer_grads)

[(Ellipsis, Ellipsis), (Ellipsis, Ellipsis)]


In [21]:
cost_grad = jax.grad(cost, argnums=0)
layer_grads_jax = cost_grad(layers, x, [sigmoid, ReLU], target)
print(layer_grads_jax)

# the check: every dW and db agrees with JAX to machine precision
print([ (np.allclose(dW, dW_j), np.allclose(db, db_j))
        for (dW, db), (dW_j, db_j) in zip(layer_grads, layer_grads_jax) ])

[(Array([[0.02795055, 0.02417766],
       [0.29173733, 0.25235735],
       [0.27633981, 0.23903825]], dtype=float64), Array([0.0326645 , 0.34093982, 0.32294545], dtype=float64)), (Array([[0.05833136, 0.42078901, 0.49798264],
       [0.02604824, 0.18790604, 0.22237734],
       [0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        ]], dtype=float64), Array([0.75167911, 0.33566713, 0.        , 0.        ], dtype=float64))]


TypeError: unsupported operand type(s) for -: 'ellipsis' and 'float'

# Exercise 6 - Batched inputs

Make new versions of all the functions in exercise 5 which now take batched
inputs instead. See last week's exercise 5 (and its solution) for details on
how to batch inputs to neural networks: samples in the rows of `inputs`,
weights of shape (inputs, outputs), `z = a @ W + b`. You will also need to
update the backpropagation function. Two things change there: the error is
propagated with `dC_dz @ W.T` instead of `W.T @ dC_dz`, and the gradients
of the weights and biases are *sums over the samples*,
`dC_dW = layer_input.T @ dC_dz` and `dC_db = np.sum(dC_dz, axis=0)` — compare
Eqs. (8.65)–(8.68) of the lecture notes written for a batch. Check your
batched backpropagation against `jax.grad` of the batched cost, as above.


# Exercise 7 - Training


**a)** Complete exercises 6 and 7 from last week (the iris data, softmax
output and cross entropy), but use your own backpropagation implementation
to compute the gradient.
- IMPORTANT: Do not implement the derivative terms for the softmax and the
  cross entropy separately, it will be very hard!
- Instead, use the fact that the derivatives multiplied together simplify to
  **prediction − target**: for the softmax output with the cross entropy the
  error of the last layer is $\delta^L = a^L - y$, Eq. (8.69) of the lecture
  notes (derived in Section [5.6](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter5.html) for the softmax; see also
  [this derivation](https://shivammehta25.github.io/posts/deriving-categorical-cross-entropy-and-softmax/)).
  In your `backpropagation` function this means that for the last layer you
  set `dC_dz = predict - target` directly, skipping `cost_der` and
  `activation_der`.

**b)** Use stochastic gradient descent with momentum when you train your
network (Sections [4.6 and 4.7](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter4.html) of the lecture notes; reuse
your Project 1 code). Call the learning rate `gamma`, as everywhere else in
the course, and compare the accuracy on a test set with that of the
full-batch gradient descent of last week.


# Exercise 8 (Optional) - Object orientation

Passing in the layers, activation functions, activation derivatives and cost
derivatives into the functions each time leads to code which is easy to
understand in isolation, but messier when used in a larger context with data
splitting, data scaling, gradient methods and so forth. Creating an object
which stores these values can lead to code which is much easier to use, and
is what we recommend for Project 2.

**a)** Write a neural network class. You are free to implement it how you
see fit, though we strongly recommend not to save any input or output values
as class attributes, nor to let the neural network class handle gradient
methods internally. Gradient methods should be handled outside, by
performing general operations on the `layer_grads` list using functions or
classes separate from the neural network.

We provide here a skeleton structure which should get you started.


In [ ]:
class NeuralNetwork:
    def __init__(
        self,
        network_input_size,
        layer_output_sizes,
        activation_funcs,
        activation_ders,
        cost_fun,
        cost_der,
    ):
        pass

    def predict(self, inputs):
        # Simple feed forward pass
        pass

    def cost(self, inputs, targets):
        pass

    def _feed_forward_saver(self, inputs):
        pass

    def compute_gradient(self, inputs, targets):
        pass

    def update_weights(self, layer_grads):
        pass

    # These last two methods are not needed in the project, but they can be nice to have!
    # The first one has a layers parameter so that you can use jax.grad on it.
    def jax_compliant_predict(self, layers, inputs):
        pass

    def jax_gradient(self, inputs, targets):
        pass

## Insights you should have gained this week — a self-check

Before you hand in, go through the list below without looking at the notes.
Each row is a question we may ask in the Tuesday session or on Project 2; the
right-hand column is the insight it tests, in one sentence.

| Ask yourself | The insight it tests |
|---|---|
| `W` has shape (10, 4) and `b` shape (10,). What are the shapes of $\partial C/\partial W$ and $\partial C/\partial b$? | The gradient of a scalar with respect to an array has the shape of that array: (10, 4) and (10,) — one number per parameter. |
| Which quantity is shared by $\partial C/\partial W$ and $\partial C/\partial b$ of the same layer? | The error $\delta = \partial C/\partial z = (\partial C/\partial a)\, f'(z)$; the bias gradient *is* $\delta$, the weight gradient is $\delta$ times the layer input (an outer product, Eqs. 8.61–8.63). |
| Why does `mse_der` carry a factor $2/n$? | Because `mse` takes the *mean*: $C = \frac1n\sum_i (a_i - y_i)^2$, so $\partial C/\partial a_i = \frac{2}{n}(a_i - y_i)$; forgetting the normalisation gives a gradient that is right up to a constant, which the learning rate then silently absorbs. |
| How does the error of layer $l+1$ become the error of layer $l$? | $\delta^{l} = (W^{l+1})^T \delta^{l+1} \odot f'(z^l)$ (Eq. 8.64): multiply by the transposed weights of the *next* layer and by the derivative of the activation of *this* layer. |
| Why must the forward pass save all the `z`s and `a`s? | Every derivative in the backward sweep is evaluated at the values of the forward pass: $f'(z^l)$ needs $z^l$ and the weight gradient needs the layer input $a^{l-1}$. |
| Why does the backward loop run from the last layer to the first? | The error of a layer depends on the error of the layer after it; the cost is only directly accessible at the output. |
| What changes in the batched version of backpropagation? | The error propagates as `dC_dz @ W.T` (samples stay in rows), and the parameter gradients are sums over the samples: `layer_input.T @ dC_dz` and `dC_dz.sum(axis=0)`. |
| With the softmax output and the cross entropy, what is $\delta^L$? | Simply $a^L - y$ (Eq. 8.69): the softmax Jacobian and the cross-entropy derivative cancel, which is why one never implements them separately. |
| `jax.grad` and your `backpropagation` agree to $10^{-15}$. What has JAX done? | Reverse-mode automatic differentiation of the forward pass: the same chain rule, in the same order, at the same cost of about two forward passes (Section 4.14). |
| Your network trains with full-batch gradient descent. Why switch to SGD with momentum? | One update per minibatch instead of per pass over the data, cheaper per step and with noise that helps escape flat regions; momentum damps the oscillations (Sections 4.6–4.7). |


### Practicalities

Deliver your solutions (a single Jupyter notebook) in Canvas before
**Friday October 16 at midnight**. Collaboration is encouraged — list your
collaborators. The backpropagation code you write this week is the core of
Project 2 (deadline November 6).

**Declaration of the use of large language models (LLMs).** Many of you use
ChatGPT, Claude, Copilot or similar tools when solving the weekly exercises,
and that is fine — but it has to be declared, for the weekly exercises as
well as for the projects. Add a short paragraph at the end of your notebook
(the cell below is a template) stating which tools you used and for what
(code, debugging, explanations, text), or that you did not use any. A few
sentences are enough; the full guidelines, which you must follow in the
project reports, are at
[LLM_Usage_Declaration_Guidelines.md](https://github.com/EducationalMaterialUiO/MachineLearningUiO/blob/main/LLM_Usage_Declaration_Guidelines.md). Remember that *you* are
responsible for every line you hand in: be prepared to explain it.


## Declaration of the use of LLMs

*(Replace this text with your own declaration, or write "No LLMs were used".)*

Tools used: … (for example ChatGPT, Claude, GitHub Copilot)

What they were used for: … (for example debugging the shapes in exercise 5,
explaining the chain rule in exercise 3, polishing this text)

What was done without them: …
